In [4]:
import requests

OPENALEX_API = "https://api.openalex.org/works"

def get_citation_count_openalex(doi_or_arxiv_id):
    try:
        # OpenAlex에서는 arXiv ID도 지원함. 하지만 DOI를 쓰는 게 더 정확함.
        response = requests.get(f"{OPENALEX_API}/https://doi.org/{doi_or_arxiv_id}")
        
        if response.status_code == 200:
            data = response.json()
            return data.get("cited_by_count", None)  # 인용 수 반환
        else:
            print(f"Request failed with status {response.status_code}: {response.text}")
            return None
    except requests.exceptions.RequestException as e:
        print(f"Request error: {e}")
        return None

# 테스트 실행 (DOI 사용, arXiv ID도 가능)
print(get_citation_count_openalex("10.1109/5.771073"))


113


In [5]:
import requests

CROSSREF_API = "https://api.crossref.org/works/"

def get_citation_count_crossref(doi):
    try:
        response = requests.get(f"{CROSSREF_API}{doi}")
        if response.status_code == 200:
            data = response.json()
            return data["message"].get("is-referenced-by-count", None)  # 인용 수 반환
        else:
            print(f"Request failed with status {response.status_code}: {response.text}")
            return None
    except requests.exceptions.RequestException as e:
        print(f"Request error: {e}")
        return None

# 테스트 실행 (DOI 필수)
print(get_citation_count_crossref("10.1109/5.771073"))  # 예제 DOI


45


### 구글 스콜라 크롤링 이용

In [11]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
import time

def get_citation_count_scholar_selenium(query):
    try:
        search_url = f"https://scholar.google.com/scholar?q={query.replace(' ', '+')}"
        print(f"Requesting: {search_url}")

        # Chrome Headless 설정 (UI 없이 실행)
        chrome_options = Options()
        chrome_options.add_argument("--headless")  # GUI 없이 실행
        chrome_options.add_argument("--no-sandbox")
        chrome_options.add_argument("--disable-dev-shm-usage")

        # WebDriver 실행
        driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options)
        driver.get(search_url)
        time.sleep(3)  # 페이지 로딩 대기

        # "Cited by" 텍스트가 포함된 링크 찾기
        citation_count = None
        cited_by_elements = driver.find_elements("xpath", "//a[contains(text(), 'Cited by')]")

        if cited_by_elements:
            citation_text = cited_by_elements[0].text  # 예: "Cited by 1203"
            citation_count = int(citation_text.split()[-1])

        driver.quit()
        return citation_count

    except Exception as e:
        print(f"Error fetching citation count: {e}")
        return None

# 테스트 실행
print(get_citation_count_scholar_selenium(""))


Requesting: https://scholar.google.com/scholar?q=Toward+unique+identifiers
None
